In [1]:
!pip install gymnasium[box2d]
!pip install stable-baselines3
!pip install shimmy

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------------------------------- -------- 1.0/1.3 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 4.5 MB/s  0:00:00
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 1.3 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 1.3 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.5 MB 1.3 MB/s eta 0:00:02
   ------------------------ --------------- 1.6/2.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.5 MB 1.3 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.5 MB 1.3 MB/s eta 0:00:01
   ------------------------------------- -- 2.4/2.5 MB 1.2 MB/s eta 0:00:01
   ---------------------------------------

In [2]:
import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

import os

In [3]:
env = gym.make("LunarLander-v3")

print("Observation Space:", env.observation_space)
print("Action Space:", env.action_space)

Observation Space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action Space: Discrete(4)


In [4]:
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    gamma=0.99
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [5]:
model.learn(total_timesteps=100000)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.5     |
|    ep_rew_mean     | -162     |
| time/              |          |
|    fps             | 1527     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 97.8         |
|    ep_rew_mean          | -174         |
| time/                   |              |
|    fps                  | 936          |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0074609695 |
|    clip_fraction        | 0.0191       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | 0.00471      |
|    learning_r

In [6]:
os.makedirs("models", exist_ok=True)

In [7]:
model.save("models/lunar_lander_ppo")
print("Model saved successfully!")

Model saved successfully!


In [8]:
loaded_model = PPO.load("models/lunar_lander_ppo")

In [9]:
mean_reward, std_reward = evaluate_policy(
    loaded_model,
    env,
    n_eval_episodes=10,
    deterministic=True
)

print("Mean Reward:", mean_reward)
print("Standard Deviation:", std_reward)

c:\Users\shris\OneDrive\Desktop\Lunar-Lander-RL-Agent\.venv\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Mean Reward: -93.66038537122077
Standard Deviation: 76.24901995411676


In [10]:
test_env = gym.make("LunarLander-v3", render_mode="human")

obs, info = test_env.reset()

for _ in range(1000):
    action, _ = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = test_env.step(action)

    if terminated or truncated:
        obs, info = test_env.reset()

test_env.close()

In [11]:
for episode in range(5):

    obs, info = env.reset()
    done = False
    total_reward = 0

    while not done:
        action, _ = loaded_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)

        total_reward += reward
        done = terminated or truncated

    print(f"Episode {episode+1}: Reward = {total_reward:.2f}")

Episode 1: Reward = -43.66
Episode 2: Reward = -168.21
Episode 3: Reward = -253.81
Episode 4: Reward = -179.72
Episode 5: Reward = -44.91


In [12]:
env.close()
print("Environment Closed Successfully")

Environment Closed Successfully
